# 03 — MARL Training (Google Colab)

Trains the price agent and the sentiment agent for **one training seed** (`SEED`, default 0)
for `training.total_timesteps` steps each (see `config.yaml`).

Checkpoints are saved every `training.save_frequency` steps to `results/checkpoints_scaled/seed{SEED}/`
(`paths.checkpoints_root` in `config.yaml`) together with `training_setup.json`, and copied to `MyDrive/MARL/results/checkpoints_scaled/seed{SEED}/` as soon as each one is written.

**Price-agent fix (TASK5), used by both agents**
- Price features are standardised with the train-split mean/std in `data/price_scaler.json`
  (comes with the git clone) and clipped to +-5, inside the environment, so training,
  evaluation, the coordinator and SHAP all see the same scaled features.
- Each agent's action is mapped to weights exactly as in evaluation (clip to [0, 1], divide by the sum).
- While one agent trains, the other is a fixed partner, `training.partner` in `config.yaml`
  (`neutral` = equal weights, default).
- Checkpoints from before the fix (the old `checkpoints/seed{SEED}/` folder) are never resumed:
  training refuses a folder whose `training_setup.json` does not match the current setup,
  and evaluation / SHAP refuse to load such a folder too.
  Download `training_setup.json` together with the `_final.zip` files.

**Before running**
1. Push the repo (current `config.yaml`, `src/`, `data/price_scaler.json`).
2. On Drive: `MyDrive/MARL/data/processed/{TICKER}_processed.csv` and
   `MyDrive/MARL/data/sentiment/{TICKER}_sentiment.csv` (written by notebook 05) for every ticker.

Train seed 0 first, run the policy check at the end, and stop. Change `SEED` and re-run only after checking.

## Step 1 — Training seed

In [ ]:
SEED = 0

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/MARL"
print("Drive mounted.")

## Step 3 — Clone repo and install dependencies

In [ ]:
import os

if not os.path.exists("/content/MARL"):
    !git clone https://github.com/lww637-cmyk/MARL.git /content/MARL
else:
    !git -C /content/MARL pull
%cd /content/MARL
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

import sys
sys.path.insert(0, "/content/MARL")
print("Dependencies installed.")

## Step 4 — Copy data files from Drive
Fails if any processed or sentiment file is missing, so training never runs without sentiment.

In [ ]:
import shutil
import pandas as pd
from src.training.train_marl import load_config

config  = load_config()
TICKERS = config["data"]["tickers"]
print("Tickers:", TICKERS, "| end_date:", config["data"]["end_date"])

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/sentiment", exist_ok=True)

missing = []
for t in TICKERS:
    for kind in ["processed", "sentiment"]:
        src = f"{DRIVE}/data/{kind}/{t}_{kind}.csv"
        if not os.path.exists(src):
            missing.append(src)
            continue
        shutil.copy(src, f"data/{kind}/{t}_{kind}.csv")
assert not missing, f"Missing on Drive: {missing}"

for t in TICKERS:
    p = pd.read_csv(f"data/processed/{t}_processed.csv", parse_dates=["date"])
    s = pd.read_csv(f"data/sentiment/{t}_sentiment.csv", parse_dates=["date"])
    print(f"{t}: prices {p.date.min().date()}..{p.date.max().date()} ({len(p)} rows) | "
          f"sentiment {s.date.min().date()}..{s.date.max().date()} ({len(s)} days)")

## Step 5 — Validate environment (train split)
Also checks the price scaler: scaled train-split price features should have mean about 0, std about 1, and stay within +-5.

In [ ]:
import numpy as np
from src.environment.stock_env import StockTradingEnv, PRICE_FEATURES

env = StockTradingEnv(config, mode="train", split="train")
print("Partner while training:", config["training"].get("partner", "neutral"))
sc = env.price_scaler
print(f"Price scaler: {sc['n_days']} train days ({sc['train_start']} .. {sc['train_end']})")
n_price = len(PRICE_FEATURES) * len(TICKERS)
obs = []
for i in range(env.n_steps):
    env.start_idx, env.timestep = i, 0
    obs.append(env.observe("price_agent")[:n_price])
obs = np.array(obs)
print(f"Scaled price features: min {obs.min():.2f} max {obs.max():.2f} "
      f"mean {obs.mean():.3f} std {obs.std():.3f}")
assert np.abs(obs).max() <= sc["clip"] and abs(obs.mean()) < 0.05

print(f"Train split: {env.n_steps} trading days "
      f"({env.common_dates[0].date()} .. {env.common_dates[-1].date()})")
print("Price obs:", env.observation_space("price_agent").shape,
      "| Sentiment obs:", env.observation_space("sentiment_agent").shape)

for t in TICKERS:
    d = env.dataset[t].loc[env.common_dates]
    print(f"  {t}: has_sentiment on {d.has_sentiment.mean()*100:.1f}% of train days")
    assert d.has_sentiment.mean() > 0, f"{t} has no sentiment in the train split"

## Step 6 — Train one seed (resumes automatically)
Every checkpoint is copied to Drive as soon as it is saved. If Colab disconnects, re-run the notebook from the top: each agent continues from its newest checkpoint in `MyDrive/MARL/results/checkpoints_scaled/seed{SEED}/`, and an agent with a `_final.zip` there is skipped.

In [ ]:
from src.training.train_marl import find_checkpoints

DRIVE_CKPT = f"{DRIVE}/results/checkpoints_scaled/seed{SEED}"
TOTAL      = config["training"]["total_timesteps"]

for agent_id in ["price_agent", "sentiment_agent"]:
    found = find_checkpoints(agent_id, [DRIVE_CKPT], TOTAL)
    if found:
        step, path = found[0]
        print(f"{agent_id}: will resume from step {step:,} of {TOTAL:,} ({os.path.basename(path)})")
    else:
        print(f"{agent_id}: no checkpoint on Drive — starts from step 0")

In [ ]:
from src.training.train_marl import train

price_agent, sentiment_agent = train(config, seed=SEED, copy_dir=DRIVE_CKPT)
print("Training complete. Checkpoints on Drive:")
print(sorted(os.listdir(DRIVE_CKPT)))

## Step 7 — Policy check (train split only), then STOP
Deterministic weights of each final model on every train-split day (same scaled observations and action mapping as training). A per-asset std near 0 means the policy ignores its inputs (e.g. a constant uniform allocation).

In [ ]:
from stable_baselines3 import PPO
from src.environment.stock_env import normalise_action
from src.checkpoints import checkpoint_dir, check_setup, training_setup

CKPT = checkpoint_dir(config, SEED)
check_setup(CKPT, training_setup(config))

env = StockTradingEnv(config, mode="eval", split="train")
env.reset(seed=0)
cols = TICKERS + ["cash"]

for agent_id in ["price_agent", "sentiment_agent"]:
    model = PPO.load(f"{CKPT}/{agent_id}_final.zip")
    weights = []
    for i in range(env.n_steps):
        env.start_idx, env.timestep = i, 0
        a, _ = model.predict(env.observe(agent_id), deterministic=True)
        weights.append(normalise_action(a))
    w = pd.DataFrame(weights, columns=cols)
    print(f"\n{agent_id} — allocation over {len(w)} train days")
    print(pd.DataFrame({"mean": w.mean(), "std": w.std()}).round(3).T.to_string())

**Stop here.** Check the policy above before training more seeds. For the next seed, set `SEED` in Step 1 and re-run from Step 1.